# Experiment 2.1.3 — Temporal readout ablation

## Research question

With the same single-timescale feed-forward SNN and the same latent output representation, how should short-duration SNN features be integrated into a final letter prediction?

The SNN output layer is treated as a **latent temporal feature layer**, not as 12 class neurons.

\[
\boxed{\text{events}\rightarrow\text{FF-SNN local features}\rightarrow\text{temporal readout}\rightarrow\text{letter}}
\]

This experiment fixes:

\[
C \rightarrow 128 \rightarrow 128 \rightarrow M,\qquad M=64
\]

and compares four readouts:

| Readout | Temporal bins | Sequence integration | Uses `valid_length`? |
|---|---|---|---:|
| `relative10_linear` | 10 relative-progress bins | flatten + Linear | Yes |
| `relative10_gru` | 10 relative-progress bins | GRU + Linear | Yes |
| `fixed200_linear` | fixed ~200 ms bins over full window | flatten + Linear | No |
| `fixed200_zero_hold_gru` | fixed ~200 ms bins over full window | zero-on-hold GRU + Linear | No |

### Key controls

- All four readouts use the same SNN architecture and paired SNN initialization for a given master seed.
- `relative10_*` uses the true valid gesture duration only to construct 10 relative-progress bins.
- `relative10_gru` uses a **standard GRU update for all 10 relative bins**; it does not use zero-on-hold.
- `fixed200_*` never uses the true gesture boundary in its readout or classification loss.
- `fixed200_zero_hold_gru` updates only when a fixed bin contains latent output spikes; an all-zero bin holds the previous hidden state.
- Both GRU variants use the same `GRUCell(M, H)` and `Linear(H, K)` capacity.

Main matched comparisons:

1. `relative10_linear` vs `relative10_gru`: flattened vs sequential integration under the same relative representation.
2. `fixed200_linear` vs `fixed200_zero_hold_gru`: flattened vs causal integration under the same boundary-free fixed-duration representation.
3. `relative10_gru` vs `fixed200_zero_hold_gru`: boundary-aware relative phase vs boundary-free absolute time with the same GRU capacity.


In [ ]:
from __future__ import annotations

from pathlib import Path
import copy
import hashlib
import json
import math
import os
import random
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from IPython.display import display

import snntorch as snn
from snntorch import surrogate
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "snn").is_dir() and (candidate / "notebooks").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate the writingRing repository root")


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from snn.accel_reconstruction_eval.datasets import load_acceleration_data

print("Repository root:", REPO_ROOT)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())


## 1. Configuration


In [ ]:
DATASET_ROOTS = [
    REPO_ROOT / "outputs/action0_wavelets_0e5_1_2_4_8_sr_64",
    REPO_ROOT / "outputs/action1_wavelets_0e5_1_2_4_8_sr_64",
]

INCLUDED_LABELS = (
    "A", "B", "C", "D", "E", "X",
    "G", "H", "I", "J", "K", "L",
)

SPLIT_SEED = 12345
TRAIN_FRACTION = 0.70
VAL_FRACTION = 0.15

SEEDS = (11, 23, 101)

READOUTS = (
    "relative10_linear",
    "relative10_gru",
    "fixed200_linear",
    "fixed200_zero_hold_gru",
)

READOUT_USES_VALID_LENGTH = {
    "relative10_linear": True,
    "relative10_gru": True,
    "fixed200_linear": False,
    "fixed200_zero_hold_gru": False,
}

N_RELATIVE_BINS = 10
FIXED_BIN_MS = 200.0

HIDDEN_WIDTH = 128
OUTPUT_WIDTH = 64
GRU_HIDDEN_WIDTH = 32

# Single hidden-layer tau_syn setting; change this value to test another single scale.
HIDDEN_SHIFT_SYN = 2

TAU_MEM_MS = 22.54
TAU_SYN_OUT_MS = 77.47
THRESHOLD = 0.5
SURROGATE_SLOPE = 25.0
RESET_MECHANISM = "subtract"

SPIKE_REGULARIZATION = 0.0
FIRING_ONSET_THRESHOLD = 1e-3
ZERO_INPUT_DIAGNOSTIC_STEPS = 200

BATCH_SIZE = 128
NUM_WORKERS = 0
NUM_EPOCHS = 100
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 0.0
GRAD_CLIP_NORM = None

RESUME_EXISTING = True

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

EXPERIMENT_ID = "experiment_2_1_3_temporal_readout_ablation"
RESULTS_DIR = REPO_ROOT / "notebooks/artifacts" / EXPERIMENT_ID
EXPECTED_TRAINING_RUNS = len(READOUTS) * len(SEEDS)

print("Experiment:", EXPERIMENT_ID)
print("Readouts:", READOUTS)
print("Uses valid length:", READOUT_USES_VALID_LENGTH)
print("Seeds:", SEEDS)
print("Fixed split seed:", SPLIT_SEED)
print("Hidden width:", HIDDEN_WIDTH)
print("Latent output width:", OUTPUT_WIDTH)
print("GRU hidden width:", GRU_HIDDEN_WIDTH)
print("Hidden shift_syn:", HIDDEN_SHIFT_SYN)
print("Full trainings:", EXPECTED_TRAINING_RUNS)
print("Epoch budget:", NUM_EPOCHS)
print("Device:", DEVICE)
print("Resume existing:", RESUME_EXISTING)


## 2. Reproducibility and shift helpers


In [ ]:
def derive_seed(master_seed: int, *parts: object) -> int:
    text = "|".join([str(master_seed), *(str(p) for p in parts)])
    digest = hashlib.sha256(text.encode("utf-8")).digest()
    return int.from_bytes(digest[:4], "little", signed=False)


def seed_everything(seed: int) -> None:
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    try:
        torch.use_deterministic_algorithms(True, warn_only=True)
    except TypeError:
        torch.use_deterministic_algorithms(True)


def worker_init_fn(worker_id: int) -> None:
    worker_seed = torch.initial_seed() % (2**32)
    random.seed(worker_seed)
    np.random.seed(worker_seed)


def shift_to_alpha(shift: int) -> float:
    return float(1.0 - 2.0 ** (-int(shift)))


def alpha_to_tau_ms(alpha: float, sampling_rate_hz: float) -> float:
    dt_ms = 1000.0 / float(sampling_rate_hz)
    return float(-dt_ms / math.log(float(alpha)))


## 3. Load and validate unsigned event data


In [ ]:
data = load_acceleration_data(
    DATASET_ROOTS,
    repository_root=REPO_ROOT,
    require_reconstruction=False,
)

sampling_rates = {float(m.sampling_rate_hz) for m in data.producer_metadatas}
if len(sampling_rates) != 1:
    raise ValueError(f"Expected one shared sampling rate, got {sampling_rates}")
SAMPLING_RATE_HZ = sampling_rates.pop()

event_contracts = set()
for metadata in data.producer_metadatas:
    raw = metadata.raw
    event_representation = raw.get("event_representation")
    event_feature_schema = raw.get("event_feature_schema")
    event_channel_count = raw.get("event_channel_count")
    encoder_spec_sha256 = raw.get("spike_encoder_spec_sha256")

    if event_representation != "unsigned":
        raise ValueError(
            "Requires event_representation='unsigned'; "
            f"got {event_representation!r}"
        )
    if not isinstance(event_feature_schema, str) or not event_feature_schema:
        raise ValueError("Missing event_feature_schema")
    if not isinstance(event_channel_count, int) or event_channel_count <= 0:
        raise ValueError("event_channel_count must be positive")
    if event_channel_count != metadata.channel_count - 6:
        raise ValueError(
            "Expected six auxiliary IMU channels after event channels; "
            f"event={event_channel_count}, total={metadata.channel_count}"
        )
    if not isinstance(encoder_spec_sha256, str) or len(encoder_spec_sha256) != 64:
        raise ValueError("Missing spike_encoder_spec_sha256")

    event_contracts.add(
        (
            event_representation,
            event_feature_schema,
            event_channel_count,
            encoder_spec_sha256,
        )
    )

if len(event_contracts) != 1:
    raise ValueError(f"Incompatible event contracts: {event_contracts}")

(
    EVENT_REPRESENTATION,
    EVENT_FEATURE_SCHEMA,
    INPUT_CHANNELS,
    ENCODER_SPEC_SHA256,
) = event_contracts.pop()

if INPUT_CHANNELS != 30:
    raise ValueError(
        f"Experiment 2.1.2 expects 30 polarity-split event channels, got {INPUT_CHANNELS}"
    )

rows = []
included = None if INCLUDED_LABELS is None else set(map(str, INCLUDED_LABELS))
padded_lengths = set()

for package_index, package in enumerate(data.packages):
    padded_lengths.add(int(package.padded_spike_imu.shape[1]))

    for segment_index, label in enumerate(package.labels.astype(str)):
        label = str(label)
        if included is not None and label not in included:
            continue

        valid_length = int(package.valid_lengths[segment_index])
        if valid_length <= 0:
            raise ValueError("valid_length must be positive")

        event_values = np.asarray(
            package.padded_spike_imu[
                segment_index, :valid_length, :INPUT_CHANNELS
            ]
        )
        if np.any(~np.isfinite(event_values)):
            raise ValueError("Non-finite event values")
        if np.any(event_values < 0.0):
            raise ValueError(
                "Unsigned event contract violated for "
                f"{package.user}/action_{package.action}/{segment_index}"
            )

        rows.append(
            {
                "package_index": package_index,
                "segment_index": segment_index,
                "user": str(package.user),
                "action": str(package.action),
                "label": label,
                "valid_length": valid_length,
                "sample_id": f"{package.user}/action_{package.action}/{segment_index}",
            }
        )

if len(padded_lengths) != 1:
    raise ValueError(f"Expected one padded length, got {padded_lengths}")
PADDED_LENGTH = padded_lengths.pop()

manifest = pd.DataFrame(rows)
if manifest.empty:
    raise ValueError("No samples remain after label filtering")

labels_sorted = sorted(manifest["label"].unique().tolist())
CLASS_TO_IDX = {label: i for i, label in enumerate(labels_sorted)}
IDX_TO_CLASS = {i: label for label, i in CLASS_TO_IDX.items()}
manifest["label_idx"] = manifest["label"].map(CLASS_TO_IDX).astype(int)
NUM_CLASSES = len(CLASS_TO_IDX)

DT_MS = 1000.0 / SAMPLING_RATE_HZ
HIDDEN_ALPHA = shift_to_alpha(HIDDEN_SHIFT_SYN)
HIDDEN_TAU_SYN_MS = alpha_to_tau_ms(HIDDEN_ALPHA, SAMPLING_RATE_HZ)
BETA = float(math.exp(-DT_MS / TAU_MEM_MS))
OUTPUT_ALPHA = float(math.exp(-DT_MS / TAU_SYN_OUT_MS))

FIXED_BIN_STEPS = max(
    1,
    int(round(FIXED_BIN_MS * SAMPLING_RATE_HZ / 1000.0)),
)
FIXED_BIN_EFFECTIVE_MS = 1000.0 * FIXED_BIN_STEPS / SAMPLING_RATE_HZ
FIXED_BIN_COUNT = int(math.ceil(PADDED_LENGTH / FIXED_BIN_STEPS))
FIXED_BIN_PADDED_STEPS = FIXED_BIN_COUNT * FIXED_BIN_STEPS

print(f"samples={len(manifest)}, users={manifest.user.nunique()}, classes={NUM_CLASSES}")
print(f"input_channels={INPUT_CHANNELS}")
print(f"sampling_rate={SAMPLING_RATE_HZ} Hz, padded_length={PADDED_LENGTH}")
print(
    f"hidden shift={HIDDEN_SHIFT_SYN}, alpha={HIDDEN_ALPHA:.6f}, "
    f"tau_syn={HIDDEN_TAU_SYN_MS:.3f} ms"
)
print(
    f"beta={BETA:.6f}, output_alpha={OUTPUT_ALPHA:.6f}, "
    f"threshold={THRESHOLD}"
)
print(
    f"fixed bin requested={FIXED_BIN_MS:.1f} ms, "
    f"implemented={FIXED_BIN_STEPS} samples="
    f"{FIXED_BIN_EFFECTIVE_MS:.3f} ms, bins={FIXED_BIN_COUNT}, "
    f"aggregation padded steps={FIXED_BIN_PADDED_STEPS}"
)


## 4. One fixed user-disjoint split


In [ ]:
def make_user_split(manifest: pd.DataFrame) -> pd.DataFrame:
    users = sorted(manifest["user"].unique().tolist())
    rng = np.random.default_rng(SPLIT_SEED)
    perm = np.array(users, dtype=object)
    rng.shuffle(perm)

    n = len(perm)
    n_train = max(1, int(np.floor(TRAIN_FRACTION * n)))
    n_val = max(1, int(np.floor(VAL_FRACTION * n)))
    if n_train + n_val >= n:
        n_train, n_val = n - 2, 1

    train_users = set(perm[:n_train].tolist())
    val_users = set(perm[n_train:n_train + n_val].tolist())

    out = manifest.copy()
    out["split"] = out["user"].map(
        lambda u: "train" if u in train_users else (
            "val" if u in val_users else "test"
        )
    )
    return out


FIXED_SPLIT_MANIFEST = make_user_split(manifest)

split_summary = FIXED_SPLIT_MANIFEST.groupby("split").agg(
    users=("user", "nunique"),
    samples=("label", "size"),
    classes_present=("label", "nunique"),
)
display(split_summary)

for split_name in ("train", "val", "test"):
    n_classes = int(
        FIXED_SPLIT_MANIFEST.loc[
            FIXED_SPLIT_MANIFEST.split == split_name,
            "label",
        ].nunique()
    )
    if n_classes != NUM_CLASSES:
        raise ValueError(
            f"Split {split_name!r} contains {n_classes}/{NUM_CLASSES} classes"
        )


## 5. Dataset and paired DataLoaders


In [ ]:
class EventSNNDataset(Dataset):
    def __init__(self, data, subset_manifest: pd.DataFrame):
        self.data = data
        self.df = subset_manifest.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        row = self.df.iloc[index]
        package = self.data.packages[int(row.package_index)]
        segment_index = int(row.segment_index)
        valid_length = int(row.valid_length)

        x = np.asarray(
            package.padded_spike_imu[
                segment_index, :, :INPUT_CHANNELS
            ],
            dtype=np.float32,
        ).copy()

        if x.shape != (PADDED_LENGTH, INPUT_CHANNELS):
            raise ValueError(f"Unexpected input shape {x.shape}")
        if np.any(x[:valid_length] < 0.0):
            raise ValueError("Unsigned event contract violated")

        # Enforce the fixed-window input contract: the padded observation tail is zero input.
        # The fixed-window objectives still aggregate ALL model outputs over PADDED_LENGTH;
        # they never mask their loss/readout by valid_length.
        x[valid_length:] = 0.0
        valid_mask = np.arange(PADDED_LENGTH) < valid_length

        return {
            "x": torch.from_numpy(x),
            "label": torch.tensor(int(row.label_idx), dtype=torch.long),
            "valid_mask": torch.from_numpy(valid_mask),
            "valid_length": torch.tensor(valid_length, dtype=torch.long),
            "sample_id": str(row.sample_id),
        }


def make_loader(
    subset_manifest: pd.DataFrame,
    *,
    batch_size: int,
    shuffle: bool,
    seed: int,
):
    dataset = EventSNNDataset(data, subset_manifest)
    generator = torch.Generator().manual_seed(seed)

    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=NUM_WORKERS,
        generator=generator,
        worker_init_fn=worker_init_fn if NUM_WORKERS > 0 else None,
        pin_memory=torch.cuda.is_available(),
    )


def make_split_loaders(master_seed: int, include_test: bool = False):
    specs = [
        ("train", "train", True),
        ("train_eval", "train", False),
        ("val", "val", False),
    ]
    if include_test:
        specs.append(("test", "test", False))

    loaders = {}
    for name, split_name, shuffle in specs:
        subset = FIXED_SPLIT_MANIFEST[
            FIXED_SPLIT_MANIFEST.split == split_name
        ]
        loaders[name] = make_loader(
            subset,
            batch_size=BATCH_SIZE,
            shuffle=shuffle,
            # Deliberately objective-independent for paired runs.
            seed=derive_seed(master_seed, name, "loader"),
        )
    return loaders


## 6. Temporal aggregation helpers

In [ ]:
def relative_temporal_bin_counts(
    output_spikes: torch.Tensor,
    valid_lengths: torch.Tensor,
    n_bins: int,
) -> torch.Tensor:
    """Return [B, n_bins, M] counts over gesture-relative valid-time bins."""
    _, time_steps, _ = output_spikes.shape
    lengths = valid_lengths.to(output_spikes.device, dtype=torch.long)

    if torch.any(lengths <= 0):
        raise ValueError("valid_lengths must be positive")

    t = torch.arange(time_steps, device=output_spikes.device)[None, :]
    valid = t < lengths[:, None]

    bin_index = torch.div(
        t * n_bins,
        lengths[:, None],
        rounding_mode="floor",
    ).clamp_max(n_bins - 1)

    one_hot = F.one_hot(
        bin_index,
        num_classes=n_bins,
    ).to(output_spikes.dtype)
    one_hot = one_hot * valid.unsqueeze(-1).to(output_spikes.dtype)

    return torch.einsum("btn,btm->bnm", one_hot, output_spikes)


def fixed_window_bin_counts(output_spikes: torch.Tensor) -> torch.Tensor:
    """Return [B, fixed_bins, M] counts over the complete fixed observation window."""
    batch_size, time_steps, feature_count = output_spikes.shape
    if time_steps != PADDED_LENGTH:
        raise ValueError(f"Expected {PADDED_LENGTH} timesteps, got {time_steps}")

    # Aggregation-only zero padding for the final partial fixed-duration bin.
    if FIXED_BIN_PADDED_STEPS > time_steps:
        output_spikes = F.pad(
            output_spikes,
            (0, 0, 0, FIXED_BIN_PADDED_STEPS - time_steps),
        )

    return output_spikes.reshape(
        batch_size,
        FIXED_BIN_COUNT,
        FIXED_BIN_STEPS,
        feature_count,
    ).sum(dim=2)


def bin_activity_mask(bin_counts: torch.Tensor) -> torch.Tensor:
    """Return [B, bins] bool mask: True iff a bin contains any latent output spike."""
    return bin_counts.abs().sum(dim=-1) > 0


## 7. Single-$\tau_{\mathrm{syn}}$ latent-output SNN and four temporal readouts

The SNN is fixed:

\[
30\rightarrow128\rightarrow128\rightarrow64.
\]

The 64 output neurons are latent temporal feature neurons.

### Relative + Linear

\[
[T,64]\rightarrow[10,64]\rightarrow\mathrm{flatten}(640)\rightarrow\mathrm{Linear}(640,12).
\]

### Relative + GRU

\[
[T,64]\rightarrow[10,64]\rightarrow\mathrm{GRUCell}(64,32)\rightarrow\mathrm{Linear}(32,12).
\]

All 10 relative bins are valid gesture-progress bins, so the GRU updates on every bin.

### Fixed 200 ms + Linear

\[
[T,64]\rightarrow[B_{fixed},64]\rightarrow\mathrm{flatten}\rightarrow\mathrm{Linear}\rightarrow12.
\]

No `valid_length` enters this readout.

### Fixed 200 ms + zero-on-hold GRU

For each bin, compute a GRU candidate state. If the bin has no latent spikes, hold the previous state instead of applying the candidate update. This prevents the padded silent tail from changing the accumulated gesture representation without using the true gesture boundary.


In [ ]:
class TemporalReadoutSNN(nn.Module):
    def __init__(
        self,
        *,
        readout: str,
        num_classes: int,
        input_channels: int,
        master_seed: int,
    ):
        super().__init__()

        if readout not in READOUTS:
            raise ValueError(f"Unknown readout: {readout}")

        self.readout = str(readout)
        self.num_classes = int(num_classes)
        self.input_channels = int(input_channels)
        self.output_width = int(OUTPUT_WIDTH)

        spike_grad = surrogate.fast_sigmoid(slope=SURROGATE_SLOPE)

        # Pair the SNN initialization across all four readouts within a master seed.
        seed_everything(derive_seed(master_seed, "fc1_init"))
        self.fc1 = nn.Linear(input_channels, HIDDEN_WIDTH, bias=False)
        self.lif1 = snn.Synaptic(
            alpha=HIDDEN_ALPHA,
            beta=BETA,
            threshold=THRESHOLD,
            spike_grad=spike_grad,
            learn_alpha=False,
            learn_beta=False,
            learn_threshold=False,
            reset_mechanism=RESET_MECHANISM,
        )

        seed_everything(derive_seed(master_seed, "fc2_init"))
        self.fc2 = nn.Linear(HIDDEN_WIDTH, HIDDEN_WIDTH, bias=False)
        self.lif2 = snn.Synaptic(
            alpha=HIDDEN_ALPHA,
            beta=BETA,
            threshold=THRESHOLD,
            spike_grad=spike_grad,
            learn_alpha=False,
            learn_beta=False,
            learn_threshold=False,
            reset_mechanism=RESET_MECHANISM,
        )

        seed_everything(derive_seed(master_seed, "latent_output_init", OUTPUT_WIDTH))
        self.fc_out = nn.Linear(HIDDEN_WIDTH, OUTPUT_WIDTH, bias=False)
        self.lif_out = snn.Synaptic(
            alpha=OUTPUT_ALPHA,
            beta=BETA,
            threshold=THRESHOLD,
            spike_grad=spike_grad,
            learn_alpha=False,
            learn_beta=False,
            learn_threshold=False,
            reset_mechanism=RESET_MECHANISM,
        )

        self.linear_head = None
        self.gru_cell = None
        self.gru_classifier = None

        if self.readout == "relative10_linear":
            seed_everything(derive_seed(master_seed, "relative10_linear_head_init"))
            self.linear_head = nn.Linear(
                N_RELATIVE_BINS * OUTPUT_WIDTH,
                num_classes,
                bias=True,
            )

        elif self.readout == "fixed200_linear":
            seed_everything(derive_seed(master_seed, "fixed200_linear_head_init"))
            self.linear_head = nn.Linear(
                FIXED_BIN_COUNT * OUTPUT_WIDTH,
                num_classes,
                bias=True,
            )

        elif self.readout in {"relative10_gru", "fixed200_zero_hold_gru"}:
            # Same GRU/classifier initialization across the two GRU variants.
            seed_everything(derive_seed(master_seed, "shared_gru_init"))
            self.gru_cell = nn.GRUCell(
                OUTPUT_WIDTH,
                GRU_HIDDEN_WIDTH,
                bias=True,
            )
            seed_everything(derive_seed(master_seed, "shared_gru_classifier_init"))
            self.gru_classifier = nn.Linear(
                GRU_HIDDEN_WIDTH,
                num_classes,
                bias=True,
            )
        else:
            raise RuntimeError(self.readout)

    @property
    def backbone_parameter_count(self) -> int:
        return sum(
            p.numel()
            for module in (self.fc1, self.fc2, self.fc_out)
            for p in module.parameters()
        )

    @property
    def readout_parameter_count(self) -> int:
        modules = [
            m for m in (self.linear_head, self.gru_cell, self.gru_classifier)
            if m is not None
        ]
        return sum(p.numel() for module in modules for p in module.parameters())

    @property
    def total_parameter_count(self) -> int:
        return sum(p.numel() for p in self.parameters())

    @property
    def spike_neuron_count(self) -> int:
        return 2 * HIDDEN_WIDTH + OUTPUT_WIDTH

    def forward(self, x, valid_mask=None):
        self.lif1.reset_mem()
        self.lif2.reset_mem()
        self.lif_out.reset_mem()

        spk1_rec, spk2_rec, out_rec = [], [], []
        for step in range(x.shape[1]):
            h1, _, _ = self.lif1(self.fc1(x[:, step, :]))
            h2, _, _ = self.lif2(self.fc2(h1))
            spk_out, _, _ = self.lif_out(self.fc_out(h2))
            spk1_rec.append(h1)
            spk2_rec.append(h2)
            out_rec.append(spk_out)

        return {
            "hidden_spikes": [
                torch.stack(spk1_rec, dim=1),
                torch.stack(spk2_rec, dim=1),
            ],
            "output_spikes": torch.stack(out_rec, dim=1),
        }

    def _run_gru_sequence(
        self,
        bin_sequence: torch.Tensor,
        *,
        zero_on_hold: bool,
        return_all_logits: bool = False,
    ):
        batch_size, n_bins, _ = bin_sequence.shape
        h = torch.zeros(
            batch_size,
            GRU_HIDDEN_WIDTH,
            dtype=bin_sequence.dtype,
            device=bin_sequence.device,
        )

        activity = bin_activity_mask(bin_sequence) if zero_on_hold else None
        logits_per_bin = []

        for bin_idx in range(n_bins):
            z = bin_sequence[:, bin_idx, :]
            h_candidate = self.gru_cell(z, h)

            if zero_on_hold:
                gate = activity[:, bin_idx].unsqueeze(-1)
                h = torch.where(gate, h_candidate, h)
            else:
                h = h_candidate

            if return_all_logits:
                logits_per_bin.append(self.gru_classifier(h))

        final_logits = self.gru_classifier(h)
        if return_all_logits:
            return final_logits, torch.stack(logits_per_bin, dim=1)
        return final_logits

    def temporal_bins(self, out, valid_lengths=None):
        output_spikes = out["output_spikes"]

        if self.readout.startswith("relative10"):
            if valid_lengths is None:
                raise ValueError(f"{self.readout} requires valid_lengths")
            return relative_temporal_bin_counts(
                output_spikes,
                valid_lengths,
                N_RELATIVE_BINS,
            )

        if self.readout.startswith("fixed200"):
            # Deliberately no true gesture boundary.
            return fixed_window_bin_counts(output_spikes)

        raise RuntimeError(self.readout)

    def native_logits(self, out, valid_lengths=None, valid_mask=None):
        bins = self.temporal_bins(out, valid_lengths)

        if self.readout in {"relative10_linear", "fixed200_linear"}:
            return self.linear_head(bins.flatten(start_dim=1))

        if self.readout == "relative10_gru":
            return self._run_gru_sequence(bins, zero_on_hold=False)

        if self.readout == "fixed200_zero_hold_gru":
            return self._run_gru_sequence(bins, zero_on_hold=True)

        raise RuntimeError(self.readout)

    def logits_over_bins(self, out, valid_lengths=None):
        """Return (bins, logits_per_bin) for GRU readouts only."""
        if self.readout not in {"relative10_gru", "fixed200_zero_hold_gru"}:
            raise ValueError("logits_over_bins is defined only for GRU readouts")

        bins = self.temporal_bins(out, valid_lengths)
        _, logits_per_bin = self._run_gru_sequence(
            bins,
            zero_on_hold=(self.readout == "fixed200_zero_hold_gru"),
            return_all_logits=True,
        )
        return bins, logits_per_bin

    def training_loss(self, out, labels, valid_mask, valid_lengths):
        if READOUT_USES_VALID_LENGTH[self.readout]:
            logits = self.native_logits(
                out,
                valid_lengths=valid_lengths,
                valid_mask=valid_mask,
            )
        else:
            # Fixed readouts intentionally do not receive the true boundary.
            logits = self.native_logits(
                out,
                valid_lengths=None,
                valid_mask=None,
            )

        loss = F.cross_entropy(logits, labels)

        if SPIKE_REGULARIZATION > 0:
            output_spikes = out["output_spikes"]
            mask_f = valid_mask.unsqueeze(-1).to(output_spikes.dtype)
            valid_spikes = (
                sum((spikes * mask_f).sum() for spikes in out["hidden_spikes"])
                + (output_spikes * mask_f).sum()
            )
            valid_neuron_time = (
                valid_mask.to(output_spikes.dtype).sum().clamp_min(1)
                * self.spike_neuron_count
            )
            loss = loss + SPIKE_REGULARIZATION * valid_spikes / valid_neuron_time

        return loss


## 8. Metrics, activity diagnostics, and zero-input sanity

In [ ]:
def classification_metrics(y_true, y_pred):
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "macro_f1": float(
            f1_score(y_true, y_pred, average="macro", zero_division=0)
        ),
    }


@torch.no_grad()
def evaluate_model(model, loader):
    model.eval()

    total_loss = 0.0
    sample_count = 0
    valid_steps = 0
    full_steps = 0
    y_true, y_pred = [], []

    hidden_valid_totals = [0.0, 0.0]
    hidden_full_totals = [0.0, 0.0]
    output_valid_total = 0.0
    output_full_total = 0.0
    silent_output_samples = 0

    active_bin_total = 0.0
    active_bin_sample_count = 0
    last_active_bin_values = []

    for batch in loader:
        x = batch["x"].to(DEVICE, non_blocking=True)
        y = batch["label"].to(DEVICE, non_blocking=True)
        mask = batch["valid_mask"].to(DEVICE, non_blocking=True)
        lengths = batch["valid_length"].to(DEVICE, non_blocking=True)

        out = model(x, mask)
        loss = model.training_loss(out, y, mask, lengths)

        if READOUT_USES_VALID_LENGTH[model.readout]:
            logits = model.native_logits(out, valid_lengths=lengths, valid_mask=mask)
            bins = model.temporal_bins(out, lengths)
        else:
            logits = model.native_logits(out, valid_lengths=None, valid_mask=None)
            bins = model.temporal_bins(out, None)

        pred = logits.argmax(dim=1)

        batch_size = len(y)
        batch_valid_steps = int(mask.sum().item())
        batch_full_steps = batch_size * out["output_spikes"].shape[1]

        total_loss += float(loss.item()) * batch_size
        sample_count += batch_size
        valid_steps += batch_valid_steps
        full_steps += batch_full_steps

        y_true.extend(y.cpu().tolist())
        y_pred.extend(pred.cpu().tolist())

        mask_f = mask.unsqueeze(-1).to(out["output_spikes"].dtype)

        for layer_idx, spikes in enumerate(out["hidden_spikes"]):
            hidden_valid_totals[layer_idx] += float((spikes * mask_f).sum().item())
            hidden_full_totals[layer_idx] += float(spikes.sum().item())

        batch_out_valid = (out["output_spikes"] * mask_f).sum(dim=(1, 2))
        batch_out_full = out["output_spikes"].sum(dim=(1, 2))

        output_valid_total += float(batch_out_valid.sum().item())
        output_full_total += float(batch_out_full.sum().item())
        silent_output_samples += int((batch_out_full == 0).sum().item())

        bin_active = bin_activity_mask(bins)
        active_bin_total += float(bin_active.sum().item())
        active_bin_sample_count += batch_size

        for row in bin_active:
            indices = torch.nonzero(row, as_tuple=False).flatten()
            last_active_bin_values.append(
                -1 if len(indices) == 0 else int(indices[-1].item())
            )

    metrics = classification_metrics(y_true, y_pred)
    metrics["loss"] = total_loss / max(sample_count, 1)

    for layer_idx in range(2):
        metrics[f"hidden_{layer_idx + 1}_firing_rate"] = (
            hidden_valid_totals[layer_idx]
            / max(valid_steps * HIDDEN_WIDTH, 1)
        )
        metrics[f"hidden_{layer_idx + 1}_full_window_firing_rate"] = (
            hidden_full_totals[layer_idx]
            / max(full_steps * HIDDEN_WIDTH, 1)
        )

    metrics["output_firing_rate"] = (
        output_valid_total / max(valid_steps * OUTPUT_WIDTH, 1)
    )
    metrics["output_full_window_firing_rate"] = (
        output_full_total / max(full_steps * OUTPUT_WIDTH, 1)
    )
    metrics["silent_output_fraction"] = (
        silent_output_samples / max(sample_count, 1)
    )
    metrics["mean_active_bins"] = (
        active_bin_total / max(active_bin_sample_count, 1)
    )
    metrics["mean_last_active_bin"] = (
        float(np.mean(last_active_bin_values))
        if last_active_bin_values else np.nan
    )
    metrics["samples"] = sample_count
    return metrics


@torch.no_grad()
def zero_input_activity(model, steps=ZERO_INPUT_DIAGNOSTIC_STEPS):
    model.eval()
    x = torch.zeros(1, steps, INPUT_CHANNELS, device=DEVICE)
    mask = torch.ones(1, steps, dtype=torch.bool, device=DEVICE)
    out = model(x, mask)
    return {
        "zero_hidden_1_firing_rate": float(out["hidden_spikes"][0].mean().item()),
        "zero_hidden_2_firing_rate": float(out["hidden_spikes"][1].mean().item()),
        "zero_output_firing_rate": float(out["output_spikes"].mean().item()),
    }


## 9. Fixed-budget training and best-validation-BA checkpoint

In [ ]:
def _cpu_state_dict(model):
    return {
        key: value.detach().cpu().clone()
        for key, value in model.state_dict().items()
    }


def train_one_readout(
    *,
    readout: str,
    master_seed: int,
    train_loader,
    train_eval_loader,
    val_loader,
):
    model = TemporalReadoutSNN(
        readout=readout,
        num_classes=NUM_CLASSES,
        input_channels=INPUT_CHANNELS,
        master_seed=master_seed,
    ).to(DEVICE)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
    )

    best_state = None
    best_val_ba = -np.inf
    best_val_loss = np.inf
    best_epoch = -1
    firing_onset_epoch = None
    history_rows = []

    zero_before = zero_input_activity(model)

    for epoch in range(1, NUM_EPOCHS + 1):
        model.train()

        for batch in train_loader:
            x = batch["x"].to(DEVICE, non_blocking=True)
            y = batch["label"].to(DEVICE, non_blocking=True)
            mask = batch["valid_mask"].to(DEVICE, non_blocking=True)
            lengths = batch["valid_length"].to(DEVICE, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)
            out = model(x, mask)
            loss = model.training_loss(out, y, mask, lengths)

            if not torch.isfinite(loss):
                raise FloatingPointError(
                    f"Non-finite loss for readout={readout}, seed={master_seed}"
                )

            loss.backward()
            if GRAD_CLIP_NORM is not None:
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
            optimizer.step()

        train_metrics = evaluate_model(model, train_eval_loader)
        val_metrics = evaluate_model(model, val_loader)

        if (
            firing_onset_epoch is None
            and val_metrics["output_firing_rate"] > FIRING_ONSET_THRESHOLD
        ):
            firing_onset_epoch = epoch

        val_ba = val_metrics["balanced_accuracy"]
        val_loss = val_metrics["loss"]
        improved = (
            val_ba > best_val_ba + 1e-12
            or (
                abs(val_ba - best_val_ba) <= 1e-12
                and val_loss < best_val_loss - 1e-12
            )
        )

        if improved:
            best_val_ba = val_ba
            best_val_loss = val_loss
            best_epoch = epoch
            best_state = _cpu_state_dict(model)

        history_rows.append({
            "readout": readout,
            "seed": master_seed,
            "epoch": epoch,
            "train_loss": train_metrics["loss"],
            "train_accuracy": train_metrics["accuracy"],
            "train_balanced_accuracy": train_metrics["balanced_accuracy"],
            "train_macro_f1": train_metrics["macro_f1"],
            "train_hidden_1_firing_rate": train_metrics["hidden_1_firing_rate"],
            "train_hidden_2_firing_rate": train_metrics["hidden_2_firing_rate"],
            "train_output_firing_rate": train_metrics["output_firing_rate"],
            "train_output_full_window_firing_rate": train_metrics["output_full_window_firing_rate"],
            "train_mean_active_bins": train_metrics["mean_active_bins"],
            "val_loss": val_metrics["loss"],
            "val_accuracy": val_metrics["accuracy"],
            "val_balanced_accuracy": val_metrics["balanced_accuracy"],
            "val_macro_f1": val_metrics["macro_f1"],
            "val_hidden_1_firing_rate": val_metrics["hidden_1_firing_rate"],
            "val_hidden_2_firing_rate": val_metrics["hidden_2_firing_rate"],
            "val_output_firing_rate": val_metrics["output_firing_rate"],
            "val_output_full_window_firing_rate": val_metrics["output_full_window_firing_rate"],
            "val_silent_output_fraction": val_metrics["silent_output_fraction"],
            "val_mean_active_bins": val_metrics["mean_active_bins"],
            "val_mean_last_active_bin": val_metrics["mean_last_active_bin"],
        })

        if epoch == 1 or epoch % 10 == 0 or epoch == NUM_EPOCHS:
            print(
                f"epoch {epoch:>3d} | "
                f"train loss={train_metrics['loss']:.4f} "
                f"train BA={train_metrics['balanced_accuracy']:.4f} | "
                f"val loss={val_metrics['loss']:.4f} "
                f"val BA={val_metrics['balanced_accuracy']:.4f} | "
                f"L1 FR={val_metrics['hidden_1_firing_rate']:.4f} "
                f"L2 FR={val_metrics['hidden_2_firing_rate']:.4f} "
                f"out FR={val_metrics['output_firing_rate']:.4f} | "
                f"active bins={val_metrics['mean_active_bins']:.2f}"
            )

    if best_state is None:
        raise RuntimeError("No best checkpoint selected")

    model.load_state_dict(best_state)
    zero_after = zero_input_activity(model)

    return {
        "model": model,
        "history": pd.DataFrame(history_rows),
        "best_epoch": best_epoch,
        "best_val_ba": best_val_ba,
        "best_val_loss": best_val_loss,
        "firing_onset_epoch": firing_onset_epoch,
        "zero_before": zero_before,
        "zero_after": zero_after,
    }


## 10. Persistent checkpoints / resume

In [ ]:
def checkpoint_path(readout: str, master_seed: int):
    return (
        RESULTS_DIR / "checkpoints" /
        f"{readout}_seed_{master_seed}_shift_{HIDDEN_SHIFT_SYN}_"
        f"hidden_{HIDDEN_WIDTH}_out_{OUTPUT_WIDTH}_best_val_ba.pt"
    )


def history_path(readout: str, master_seed: int):
    return (
        RESULTS_DIR / "histories" /
        f"{readout}_seed_{master_seed}_shift_{HIDDEN_SHIFT_SYN}_"
        f"hidden_{HIDDEN_WIDTH}_out_{OUTPUT_WIDTH}_history.csv"
    )


def checkpoint_contract(readout: str, master_seed: int):
    return {
        "experiment_id": EXPERIMENT_ID,
        "readout": readout,
        "readout_uses_valid_length": bool(READOUT_USES_VALID_LENGTH[readout]),
        "seed": int(master_seed),
        "split_seed": int(SPLIT_SEED),
        "num_epochs": int(NUM_EPOCHS),
        "input_channels": int(INPUT_CHANNELS),
        "num_classes": int(NUM_CLASSES),
        "hidden_width": int(HIDDEN_WIDTH),
        "output_width": int(OUTPUT_WIDTH),
        "gru_hidden_width": int(GRU_HIDDEN_WIDTH),
        "hidden_shift_syn": int(HIDDEN_SHIFT_SYN),
        "hidden_alpha": float(HIDDEN_ALPHA),
        "hidden_tau_syn_ms": float(HIDDEN_TAU_SYN_MS),
        "n_relative_bins": int(N_RELATIVE_BINS),
        "fixed_bin_requested_ms": float(FIXED_BIN_MS),
        "fixed_bin_steps": int(FIXED_BIN_STEPS),
        "fixed_bin_effective_ms": float(FIXED_BIN_EFFECTIVE_MS),
        "fixed_bin_count": int(FIXED_BIN_COUNT),
        "padded_length": int(PADDED_LENGTH),
        "tau_mem_ms": float(TAU_MEM_MS),
        "tau_syn_out_ms": float(TAU_SYN_OUT_MS),
        "threshold": float(THRESHOLD),
        "sampling_rate_hz": float(SAMPLING_RATE_HZ),
        "encoder_spec_sha256": ENCODER_SPEC_SHA256,
    }


def save_completed_run(readout: str, master_seed: int, run):
    checkpoint_path(readout, master_seed).parent.mkdir(parents=True, exist_ok=True)
    history_path(readout, master_seed).parent.mkdir(parents=True, exist_ok=True)

    payload = checkpoint_contract(readout, master_seed)
    payload.update({
        "best_epoch": int(run["best_epoch"]),
        "best_val_ba": float(run["best_val_ba"]),
        "best_val_loss": float(run["best_val_loss"]),
        "firing_onset_epoch": run["firing_onset_epoch"],
        "zero_before": run["zero_before"],
        "zero_after": run["zero_after"],
        "state_dict": _cpu_state_dict(run["model"]),
    })

    torch.save(payload, checkpoint_path(readout, master_seed))
    run["history"].to_csv(history_path(readout, master_seed), index=False)


def load_completed_run(readout: str, master_seed: int):
    ckpt = checkpoint_path(readout, master_seed)
    hist = history_path(readout, master_seed)
    if not (ckpt.exists() and hist.exists()):
        return None

    payload = torch.load(ckpt, map_location="cpu", weights_only=False)
    required = checkpoint_contract(readout, master_seed)
    for key, expected in required.items():
        if payload.get(key) != expected:
            raise ValueError(
                f"Checkpoint mismatch for {ckpt}: {key}={payload.get(key)!r}, "
                f"expected {expected!r}. Delete the stale checkpoint or change RESULTS_DIR."
            )

    model = TemporalReadoutSNN(
        readout=readout,
        num_classes=NUM_CLASSES,
        input_channels=INPUT_CHANNELS,
        master_seed=master_seed,
    ).to(DEVICE)
    model.load_state_dict(payload["state_dict"])

    return {
        "model": model,
        "history": pd.read_csv(hist),
        "best_epoch": int(payload["best_epoch"]),
        "best_val_ba": float(payload["best_val_ba"]),
        "best_val_loss": float(payload["best_val_loss"]),
        "firing_onset_epoch": payload.get("firing_onset_epoch"),
        "zero_before": payload.get("zero_before", {}),
        "zero_after": payload.get("zero_after", {}),
    }


## 11. Four-readout $\times$ paired-seed sweep

Default run count:

\[
4\text{ readouts}\times3\text{ seeds}=12.
\]

The test split is evaluated only after the best validation checkpoint is selected or restored.


In [ ]:
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

run_rows = []
history_frames = []

for master_seed in SEEDS:
    print("=" * 100)
    print("MASTER SEED:", master_seed)

    for readout in READOUTS:
        print("-" * 100)
        print("READOUT:", readout)
        print("USES VALID LENGTH:", READOUT_USES_VALID_LENGTH[readout])

        loaders = make_split_loaders(master_seed, include_test=False)
        run = load_completed_run(readout, master_seed) if RESUME_EXISTING else None

        if run is None:
            run = train_one_readout(
                readout=readout,
                master_seed=master_seed,
                train_loader=loaders["train"],
                train_eval_loader=loaders["train_eval"],
                val_loader=loaders["val"],
            )
            save_completed_run(readout, master_seed, run)
        else:
            print(
                f"Loaded completed checkpoint: best epoch={run['best_epoch']}, "
                f"best val BA={run['best_val_ba']:.4f}"
            )

        model = run["model"]
        history_frames.append(run["history"])

        test_loader = make_split_loaders(master_seed, include_test=True)["test"]
        split_metrics = {
            "train": evaluate_model(model, loaders["train_eval"]),
            "val": evaluate_model(model, loaders["val"]),
            "test": evaluate_model(model, test_loader),
        }

        row = {
            "experiment": EXPERIMENT_ID,
            "readout": readout,
            "uses_valid_length": READOUT_USES_VALID_LENGTH[readout],
            "seed": master_seed,
            "best_epoch": run["best_epoch"],
            "best_val_ba": run["best_val_ba"],
            "val_loss_at_best_val_ba": run["best_val_loss"],
            "firing_onset_epoch": (
                np.nan if run["firing_onset_epoch"] is None else run["firing_onset_epoch"]
            ),
            "input_channels": INPUT_CHANNELS,
            "hidden_width": HIDDEN_WIDTH,
            "output_width": OUTPUT_WIDTH,
            "gru_hidden_width": GRU_HIDDEN_WIDTH if "gru" in readout else np.nan,
            "hidden_shift_syn": HIDDEN_SHIFT_SYN,
            "hidden_tau_syn_ms": HIDDEN_TAU_SYN_MS,
            "num_classes": NUM_CLASSES,
            "backbone_params": model.backbone_parameter_count,
            "readout_params": model.readout_parameter_count,
            "total_params": model.total_parameter_count,
            **run["zero_after"],
        }

        for split_name, metrics in split_metrics.items():
            for metric_name, value in metrics.items():
                row[f"{split_name}_{metric_name}"] = value

        run_rows.append(row)

        print(
            f"best epoch={run['best_epoch']} | "
            f"train BA={split_metrics['train']['balanced_accuracy']:.4f} | "
            f"val BA={split_metrics['val']['balanced_accuracy']:.4f} | "
            f"test BA={split_metrics['test']['balanced_accuracy']:.4f} | "
            f"out FR={split_metrics['test']['output_firing_rate']:.4f} | "
            f"active bins={split_metrics['test']['mean_active_bins']:.2f}"
        )
        print("zero-input after best checkpoint:", run["zero_after"])

        del model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

results_df = pd.DataFrame(run_rows)
histories_df = pd.concat(history_frames, ignore_index=True)

display(results_df)


## 12. Aggregate results across paired seeds

In [ ]:
summary_df = (
    results_df
    .groupby("readout", sort=False, as_index=False)
    .agg(
        uses_valid_length=("uses_valid_length", "first"),
        mean_best_val_ba=("best_val_ba", "mean"),
        sd_best_val_ba=("best_val_ba", "std"),
        mean_test_ba=("test_balanced_accuracy", "mean"),
        sd_test_ba=("test_balanced_accuracy", "std"),
        mean_test_macro_f1=("test_macro_f1", "mean"),
        sd_test_macro_f1=("test_macro_f1", "std"),
        mean_output_firing_rate=("test_output_firing_rate", "mean"),
        mean_output_full_window_firing_rate=("test_output_full_window_firing_rate", "mean"),
        mean_silent_output_fraction=("test_silent_output_fraction", "mean"),
        mean_active_bins=("test_mean_active_bins", "mean"),
        mean_last_active_bin=("test_mean_last_active_bin", "mean"),
        mean_firing_onset_epoch=("firing_onset_epoch", "mean"),
        backbone_params=("backbone_params", "first"),
        readout_params=("readout_params", "first"),
        total_params=("total_params", "first"),
    )
)

order_map = {name: i for i, name in enumerate(READOUTS)}
summary_df["_order"] = summary_df["readout"].map(order_map)
summary_df = summary_df.sort_values("_order").drop(columns="_order").reset_index(drop=True)

display(summary_df)

paired_test_ba = results_df.pivot(
    index="seed",
    columns="readout",
    values="test_balanced_accuracy",
).reindex(columns=READOUTS)

display(paired_test_ba)


## 13. Figure saving configuration


In [ ]:
FIGURE_DIR = RESULTS_DIR / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

SAVE_PNG = True
SAVE_PDF = True
FIGURE_DPI = 300


def save_figure(fig, filename_stem):
    filename_stem = str(filename_stem).replace(" ", "_")
    saved_paths = []

    if SAVE_PNG:
        path = FIGURE_DIR / f"{filename_stem}.png"
        fig.savefig(path, dpi=FIGURE_DPI, bbox_inches="tight")
        saved_paths.append(path)

    if SAVE_PDF:
        path = FIGURE_DIR / f"{filename_stem}.pdf"
        fig.savefig(path, bbox_inches="tight")
        saved_paths.append(path)

    for path in saved_paths:
        print("Saved figure:", path)

    return saved_paths


print("Figure directory:", FIGURE_DIR)


## 14. Validation balanced accuracy vs. epoch

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))

for readout in READOUTS:
    sdf = histories_df[histories_df.readout == readout]

    for seed in SEEDS:
        seed_df = sdf[sdf.seed == seed].sort_values("epoch")
        ax.plot(
            seed_df["epoch"],
            seed_df["val_balanced_accuracy"],
            alpha=0.18,
            linewidth=1.0,
        )

    grouped = sdf.groupby("epoch")["val_balanced_accuracy"]
    mean_curve = grouped.mean()
    sd_curve = grouped.std()

    line, = ax.plot(
        mean_curve.index,
        mean_curve.values,
        linewidth=2.2,
        label=readout,
    )
    ax.fill_between(
        mean_curve.index,
        mean_curve.values - sd_curve.values,
        mean_curve.values + sd_curve.values,
        alpha=0.12,
        color=line.get_color(),
    )

ax.axhline(1.0 / NUM_CLASSES, linestyle="--", label="chance BA")
ax.set_xlabel("Epoch")
ax.set_ylabel("Validation balanced accuracy")
ax.set_title(
    "Experiment 2.1.3 — Validation BA vs epoch\n"
    f"output width={OUTPUT_WIDTH}, GRU hidden={GRU_HIDDEN_WIDTH}, "
    f"hidden shift_syn={HIDDEN_SHIFT_SYN}"
)
ax.grid(True, alpha=0.25)
ax.legend()
plt.tight_layout()
save_figure(fig, "val_ba_vs_epoch")
plt.show()


## 15. Validation/test readout comparison

In [ ]:
x = np.arange(len(READOUTS), dtype=float)

val_means, val_sds, test_means, test_sds = [], [], [], []
for readout in READOUTS:
    sdf = results_df[results_df.readout == readout]
    val_means.append(sdf["best_val_ba"].mean())
    val_sds.append(sdf["best_val_ba"].std(ddof=1))
    test_means.append(sdf["test_balanced_accuracy"].mean())
    test_sds.append(sdf["test_balanced_accuracy"].std(ddof=1))

fig, ax = plt.subplots(figsize=(12, 6))
ax.errorbar(
    x - 0.10,
    val_means,
    yerr=val_sds,
    marker="o",
    linestyle="none",
    capsize=4,
    label="best validation BA",
)
ax.errorbar(
    x + 0.10,
    test_means,
    yerr=test_sds,
    marker="o",
    linestyle="none",
    capsize=4,
    label="test BA",
)
ax.axhline(1.0 / NUM_CLASSES, linestyle="--", label="chance BA")
ax.set_xticks(x, READOUTS, rotation=25, ha="right")
ax.set_ylabel("Balanced accuracy")
ax.set_xlabel("Temporal readout")
ax.set_title("Experiment 2.1.3 — Temporal readout ablation\nmean ± SD across paired seeds")
ax.grid(True, axis="y", alpha=0.25)
ax.legend()
plt.tight_layout()
save_figure(fig, "readout_val_test_ba")
plt.show()


## 16. Latent-output firing-rate trajectory

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))

for readout in READOUTS:
    sdf = histories_df[histories_df.readout == readout]
    mean_curve = sdf.groupby("epoch")["val_output_firing_rate"].mean()
    ax.plot(
        mean_curve.index,
        mean_curve.values,
        linewidth=2.0,
        label=readout,
    )

ax.axhline(
    FIRING_ONSET_THRESHOLD,
    linestyle="--",
    label="firing-onset threshold",
)
ax.set_xlabel("Epoch")
ax.set_ylabel("Mean validation latent-output firing rate over valid gesture")
ax.set_title("Latent output activity by temporal readout")
ax.grid(True, alpha=0.25)
ax.legend()
plt.tight_layout()
save_figure(fig, "val_output_firing_rate_vs_epoch")
plt.show()


## 17. Spike raster visualization

The output row contains **latent temporal feature neurons**, not class neurons.

For each readout, the best paired-seed checkpoint is visualized on the same validation sample. The true valid end is shown only for interpretation; it is not used by either fixed-duration readout.


In [ ]:
def load_best_checkpoint_model(readout: str, master_seed: int):
    loaded = load_completed_run(readout, master_seed)
    if loaded is None:
        raise FileNotFoundError(
            f"No completed checkpoint for readout={readout}, seed={master_seed}"
        )
    return loaded["model"]


def get_raster_sample(split_name="val", index=0):
    subset = FIXED_SPLIT_MANIFEST[FIXED_SPLIT_MANIFEST.split == split_name]
    dataset = EventSNNDataset(data, subset)
    return dataset[index]


@torch.no_grad()
def collect_sample_activity(model, sample):
    model.eval()
    x = sample["x"].unsqueeze(0).to(DEVICE)
    mask = sample["valid_mask"].unsqueeze(0).to(DEVICE)
    out = model(x, mask)
    return {
        "hidden_1": out["hidden_spikes"][0][0].cpu().numpy(),
        "hidden_2": out["hidden_spikes"][1][0].cpu().numpy(),
        "output": out["output_spikes"][0].cpu().numpy(),
    }


def raster_points(ax, spike_matrix):
    times, neurons = np.nonzero(spike_matrix > 0)
    if len(times):
        ax.scatter(times, neurons, s=7, marker=".")


def plot_spike_raster(model, sample, *, readout, master_seed):
    activity = collect_sample_activity(model, sample)
    valid_length = int(sample["valid_length"].item())
    total_length = int(sample["x"].shape[0])

    fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

    for ax, key, label, neuron_count in (
        (axes[0], "hidden_1", "L1", HIDDEN_WIDTH),
        (axes[1], "hidden_2", "L2", HIDDEN_WIDTH),
        (axes[2], "output", "Latent output", OUTPUT_WIDTH),
    ):
        raster_points(ax, activity[key])
        ax.axvline(valid_length, linestyle="--", linewidth=1.2)
        ax.axvspan(valid_length, total_length, alpha=0.08)
        ax.set_xlim(-1, total_length)
        ax.set_ylim(-1, neuron_count)
        ax.set_ylabel(f"{label} neuron")
        ax.grid(axis="x", alpha=0.15)

    axes[0].set_title(
        f"L1 spikes — single shift_syn={HIDDEN_SHIFT_SYN} "
        f"(tau_syn≈{HIDDEN_TAU_SYN_MS:.2f} ms)"
    )
    axes[1].set_title(
        f"L2 spikes — single shift_syn={HIDDEN_SHIFT_SYN} "
        f"(tau_syn≈{HIDDEN_TAU_SYN_MS:.2f} ms)"
    )
    axes[2].set_xlabel("Timestep")
    axes[2].set_title(f"Latent output feature spikes — M={OUTPUT_WIDTH}")

    label_idx = int(sample["label"].item())
    sample_id = sample["sample_id"]
    fig.suptitle(
        f"{readout} | seed={master_seed} | true label={IDX_TO_CLASS[label_idx]} | "
        f"{sample_id} | valid={valid_length}/{total_length}",
        y=1.01,
    )

    plt.tight_layout()
    save_figure(fig, f"raster_{readout}_seed_{master_seed}")
    plt.show()


RASTER_SPLIT = "val"
RASTER_SAMPLE_INDEX = 0
raster_sample = get_raster_sample(RASTER_SPLIT, RASTER_SAMPLE_INDEX)

for readout in READOUTS:
    best_row = (
        results_df[results_df.readout == readout]
        .sort_values(
            ["best_val_ba", "val_loss_at_best_val_ba"],
            ascending=[False, True],
        )
        .iloc[0]
    )
    best_seed = int(best_row["seed"])

    print(
        f"{readout}: raster uses seed={best_seed}, "
        f"best val BA={best_row['best_val_ba']:.4f}"
    )

    raster_model = load_best_checkpoint_model(readout, best_seed)
    plot_spike_raster(
        raster_model,
        raster_sample,
        readout=readout,
        master_seed=best_seed,
    )

    del raster_model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


## 18. GRU prediction evolution over temporal bins

For the two GRU readouts, inspect how class evidence evolves as each temporal feature bin is integrated.

- `relative10_gru`: one normal GRU update per 10% of true gesture progress.
- `fixed200_zero_hold_gru`: one update per fixed-duration bin; zero-activity bins hold the hidden state and therefore preserve the prediction.


In [ ]:
@torch.no_grad()
def prediction_evolution(model, sample):
    if model.readout not in {"relative10_gru", "fixed200_zero_hold_gru"}:
        raise ValueError("Prediction evolution is defined only for GRU readouts.")

    model.eval()
    x = sample["x"].unsqueeze(0).to(DEVICE)
    mask = sample["valid_mask"].unsqueeze(0).to(DEVICE)
    lengths = sample["valid_length"].view(1).to(DEVICE)
    out = model(x, mask)

    if READOUT_USES_VALID_LENGTH[model.readout]:
        bins, logits_per_bin = model.logits_over_bins(out, valid_lengths=lengths)
    else:
        bins, logits_per_bin = model.logits_over_bins(out, valid_lengths=None)

    probs = torch.softmax(logits_per_bin, dim=-1)[0].cpu().numpy()
    bins_np = bins[0].cpu().numpy()
    active = np.abs(bins_np).sum(axis=1) > 0
    return probs, active


def plot_prediction_evolution(model, sample, *, readout, master_seed):
    probs, active = prediction_evolution(model, sample)

    true_idx = int(sample["label"].item())
    true_probs = probs[:, true_idx]
    top_probs = probs.max(axis=1)
    top_classes = probs.argmax(axis=1)
    bin_index = np.arange(1, len(probs) + 1)

    fig, ax = plt.subplots(figsize=(12, 6))
    ax.plot(
        bin_index,
        true_probs,
        marker="o",
        linewidth=2.0,
        label=f"true class {IDX_TO_CLASS[true_idx]}",
    )
    ax.plot(
        bin_index,
        top_probs,
        marker="x",
        linestyle="--",
        linewidth=1.5,
        label="current top-class probability",
    )

    for i, is_active in enumerate(active, start=1):
        if not is_active:
            ax.axvspan(i - 0.45, i + 0.45, alpha=0.08)

    for i, class_idx in enumerate(top_classes, start=1):
        ax.text(
            i,
            min(1.0, top_probs[i - 1] + 0.035),
            IDX_TO_CLASS[int(class_idx)],
            ha="center",
            va="bottom",
            fontsize=8,
        )

    ax.set_ylim(0.0, 1.05)
    ax.set_xticks(bin_index)

    if readout == "relative10_gru":
        ax.set_xlabel("Relative-progress bin (10% each)")
    else:
        ax.set_xlabel(f"Fixed-duration bin (~{FIXED_BIN_EFFECTIVE_MS:.1f} ms each)")

    ax.set_ylabel("Class probability")
    ax.set_title(
        f"Prediction evolution — {readout} | seed={master_seed}\n"
        "shaded bins have zero latent output activity"
    )
    ax.grid(True, alpha=0.25)
    ax.legend()
    plt.tight_layout()
    save_figure(fig, f"prediction_evolution_{readout}_seed_{master_seed}")
    plt.show()


for readout in ("relative10_gru", "fixed200_zero_hold_gru"):
    best_row = (
        results_df[results_df.readout == readout]
        .sort_values(
            ["best_val_ba", "val_loss_at_best_val_ba"],
            ascending=[False, True],
        )
        .iloc[0]
    )
    best_seed = int(best_row["seed"])
    model = load_best_checkpoint_model(readout, best_seed)
    plot_prediction_evolution(
        model,
        raster_sample,
        readout=readout,
        master_seed=best_seed,
    )
    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


## 19. Save results and provenance

In [ ]:
EXPERIMENT_TITLE = "Experiment 2.1.3 - Temporal Readout Ablation"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

results_path = RESULTS_DIR / "experiment_2_1_3_results.csv"
summary_path = RESULTS_DIR / "experiment_2_1_3_readout_summary.csv"
histories_path = RESULTS_DIR / "experiment_2_1_3_all_histories.csv"
paired_path = RESULTS_DIR / "experiment_2_1_3_paired_test_ba.csv"
metadata_path = RESULTS_DIR / "experiment_2_1_3_provenance.json"

results_df.to_csv(results_path, index=False)
summary_df.to_csv(summary_path, index=False)
histories_df.to_csv(histories_path, index=False)
paired_test_ba.to_csv(paired_path)

provenance = {
    "title": EXPERIMENT_TITLE,
    "experiment_id": EXPERIMENT_ID,
    "dataset_roots": [str(path) for path in DATASET_ROOTS],
    "event_representation": EVENT_REPRESENTATION,
    "event_feature_schema": EVENT_FEATURE_SCHEMA,
    "encoder_spec_sha256": ENCODER_SPEC_SHA256,
    "input_channels": int(INPUT_CHANNELS),
    "num_classes": int(NUM_CLASSES),
    "classes": labels_sorted,
    "sampling_rate_hz": float(SAMPLING_RATE_HZ),
    "padded_length": int(PADDED_LENGTH),
    "split_seed": int(SPLIT_SEED),
    "training_seeds": list(SEEDS),
    "expected_training_runs": int(EXPECTED_TRAINING_RUNS),
    "hidden_layers": 2,
    "neurons_per_hidden_layer": [int(HIDDEN_WIDTH), int(HIDDEN_WIDTH)],
    "latent_output_width": int(OUTPUT_WIDTH),
    "gru_hidden_width": int(GRU_HIDDEN_WIDTH),
    "hidden_shift_syn": int(HIDDEN_SHIFT_SYN),
    "hidden_alpha": float(HIDDEN_ALPHA),
    "hidden_tau_syn_ms": float(HIDDEN_TAU_SYN_MS),
    "tau_mem_ms": float(TAU_MEM_MS),
    "tau_syn_output_ms": float(TAU_SYN_OUT_MS),
    "threshold": float(THRESHOLD),
    "surrogate_slope": float(SURROGATE_SLOPE),
    "reset_mechanism": RESET_MECHANISM,
    "snn_linear_bias": False,
    "learn_alpha": False,
    "learn_beta": False,
    "learn_threshold": False,
    "readouts": list(READOUTS),
    "readout_uses_valid_length": READOUT_USES_VALID_LENGTH,
    "relative10_gru_zero_on_hold": False,
    "fixed200_zero_hold_gru_zero_on_hold": True,
    "n_relative_bins": int(N_RELATIVE_BINS),
    "fixed_bin_requested_ms": float(FIXED_BIN_MS),
    "fixed_bin_steps": int(FIXED_BIN_STEPS),
    "fixed_bin_effective_ms": float(FIXED_BIN_EFFECTIVE_MS),
    "fixed_bin_count": int(FIXED_BIN_COUNT),
    "num_epochs": int(NUM_EPOCHS),
    "early_stopping": False,
    "checkpoint_selection": "maximum validation balanced accuracy; lower validation loss as tie-break",
    "batch_size": int(BATCH_SIZE),
    "learning_rate": float(LEARNING_RATE),
    "weight_decay": float(WEIGHT_DECAY),
    "spike_regularization": float(SPIKE_REGULARIZATION),
    "firing_onset_threshold": float(FIRING_ONSET_THRESHOLD),
}

with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(provenance, f, indent=2, sort_keys=True)

print("=" * 100)
print(EXPERIMENT_TITLE)
print("=" * 100)
print(
    f"Architecture: {INPUT_CHANNELS} -> {HIDDEN_WIDTH} -> {HIDDEN_WIDTH} -> "
    f"{OUTPUT_WIDTH} latent outputs -> temporal readout -> {NUM_CLASSES} classes"
)
print(
    f"Hidden shift_syn={HIDDEN_SHIFT_SYN}, tau_syn≈{HIDDEN_TAU_SYN_MS:.3f} ms"
)
print("GRU hidden width:", GRU_HIDDEN_WIDTH)
print("Readouts:", READOUTS)
print("Readout valid-length use:", READOUT_USES_VALID_LENGTH)
print("Training seeds:", SEEDS)
print("Training runs:", EXPECTED_TRAINING_RUNS)
print("Split seed:", SPLIT_SEED)
print("Saved:")
for path in (results_path, summary_path, histories_path, paired_path, metadata_path):
    print(" ", path)


## Interpretation checklist

This is a **temporal readout ablation**, not an output-width sweep.

### A. Relative representation: Linear vs GRU

`relative10_linear` vs `relative10_gru` keeps the SNN, valid boundary, and 10 relative bins fixed. A GRU gain supports sequential composition beyond a flattened linear map.

### B. Fixed-duration representation: Linear vs zero-on-hold GRU

`fixed200_linear` vs `fixed200_zero_hold_gru` keeps the SNN and fixed-duration bins fixed, and neither readout receives `valid_length`. A GRU gain indicates that fixed-duration representation may be adequate but a rigid absolute-position flatten readout is insufficient.

### C. Relative GRU vs fixed-duration GRU

Both use the same GRU hidden width and classifier capacity. The major conceptual difference is boundary-aware relative phase versus boundary-free absolute time with event-driven state hold.

If `fixed200_zero_hold_gru` approaches `relative10_gru`, that is strong evidence that a streaming-oriented readout can recover much of the benefit of relative temporal organization without oracle gesture-end information.

### Sanity checks

- Zero-input SNN firing should remain exactly zero.
- Latent-output firing should escape the silent regime without saturating.
- Check `mean_active_bins` and `mean_last_active_bin` for fixed-duration methods to verify that the padded tail is predominantly silent.
- Prediction-evolution plots should show that zero-activity tail bins leave the fixed-GRU prediction unchanged.
